# DBSCAN Clustering on House Nodes

Written By: Keltonn [A0222591J]

AI Used: Claude Sonnet 5.5 for API Lookup, autocomplete, code generation

In this notebook, we will be performing basic spatial clustering of house locations (latitude/longitude) using DBSCAN. The idea is some regions in Singapore will have higher rents due to some geographical encoded characteristics (some areas are suburbs, some are city center, some have historical significance etc)

Distance metric follows the project default: **Haversine distance in kilometres**. DBSCAN's `eps` (max neighbour distance) is therefore exposed as a tunable value in **km** and converted to radians internally (the sklearn `haversine` metric expects coordinates in radians).

Steps:
1. Load the training house dataset.
2. Scatter-plot house locations (axes bounded to Singapore).
3. Choose `eps` via a k-distance (elbow) plot.
4. Run DBSCAN with tunable `min_samples` and `eps` (min-distance).
5. Visualise the clustering output (with optional top-N cluster filtering).

In [1]:
import joblib
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
# -- Paths --
ROOT = '/Users/keltonn/Desktop/Projects/housing_rent_price/final_report'

# -- Datasets -- 
TRAIN_HOUSE_PATH = f"{ROOT}/data/df_keys/train-house.csv"
TEST_HOUSE_PATH = f"{ROOT}/data/df_keys/test-house.csv"
STATION_KEYS_PATH = f"{ROOT}/data/features/raw/raw_graph_data/station-keys.csv"
MRT_EDGES_PATH = f"{ROOT}/data/features/raw/raw_graph_data/sg-mrt-edges-weighted.csv"
CLUSTER_OUTPUT_PATH = f"{ROOT}/data/features/cleaned/static_features/location_cluster"

# -- DBScan model --
DBSCAN_PATH = f'{ROOT}/01_data_processing/01_clustering_features/models'

# Project default distance is Haversine in km. sklearn's haversine metric returns
# distances in radians on the unit sphere, so multiply by the Earth radius to get km.
EARTH_RADIUS_KM = 6371.0088

# Approximate lat/long bounding box of mainland Singapore
SG_LAT_RANGE = (1.20, 1.48)
SG_LON_RANGE = (103.60, 104)

## 1. Load the training house dataset

We load the latitude and longitude from the cleaned keys dataset from our original EDA in the folder `~/final_report/00_data_collection`. In that notebook, we already established that there are some HDB blocks in the train set that are not in test set.

In [2]:
train_house = pd.read_csv(TRAIN_HOUSE_PATH)[['TOWN','BLOCK','STREET','POSTAL_CODE','LATITUDE','LONGITUDE']].drop_duplicates()
print(f"Loaded {len(train_house):,} rows, {train_house.shape[1]} columns")
train_house[["LATITUDE", "LONGITUDE"]].describe()

Loaded 9,406 rows, 6 columns


,LATITUDE,LONGITUDE
count,9406.000000,9406.000000
mean,1.366637,103.841101
std,0.041517,0.074150
min,1.270380,103.685228
25%,1.338785,103.774129
50%,1.364127,103.846468
75%,1.392596,103.899874
max,1.457071,103.987805


In [3]:
test_house = pd.read_csv(TEST_HOUSE_PATH)[['TOWN','BLOCK','STREET','POSTAL_CODE','LATITUDE','LONGITUDE']].drop_duplicates()
print(f"Loaded {len(test_house):,} rows, {test_house.shape[1]} columns")
test_house[["LATITUDE", "LONGITUDE"]].describe()

Loaded 8,940 rows, 6 columns


,LATITUDE,LONGITUDE
count,8940.000000,8940.000000
mean,1.366407,103.841318
std,0.041463,0.074057
min,1.270380,103.685228
25%,1.338582,103.774309
50%,1.363795,103.847069
75%,1.392388,103.900078
max,1.457071,103.987805


## 2. Scatter plot of house locations

Here I had Claude plot out the distribution of houses in the training dataset. The houses generally form a nice image of Singapore, and just by looking at the plot we can already see that the houses form a nice set of clusters.

There is a set of points that looks like an outlier at the coordinates (1.388547N, 103.987805E), but we verified the location on Google maps and its legit. Changi Village has some rentals and it is in a very obscure location.

In [4]:
train_house.sort_values('LONGITUDE')

,TOWN,BLOCK,STREET,POSTAL_CODE,LATITUDE,LONGITUDE
1150,jurong west,906,jurong west street 91,640906,1.341221,103.685228
28403,jurong west,905,jurong west street 91,640905,1.340470,103.685234
13101,jurong west,907,jurong west street 91,640907,1.341140,103.685931
590,jurong west,904,jurong west street 91,640904,1.340228,103.685937
1852,jurong west,903,jurong west street 91,640903,1.339717,103.686061
...,...,...,...,...,...,...
853,pasir ris,272,pasir ris street 21,510272,1.364657,103.964722
22577,pasir ris,275,pasir ris street 21,510275,1.364472,103.964915
8314,pasir ris,5,changi village road,500005,1.388853,103.986846
11536,pasir ris,4,changi village road,500004,1.389096,103.987463


In [5]:
fig = px.scatter(
    train_house,
    x="LONGITUDE",
    y="LATITUDE",
    opacity=0.4,
    title="Training House Locations (Singapore)",
    labels={"LONGITUDE": "Longitude", "LATITUDE": "Latitude"},
)
fig.update_traces(marker=dict(size=4))
fig.update_xaxes(range=list(SG_LON_RANGE))
fig.update_yaxes(range=list(SG_LAT_RANGE))
# Keep 1:1 aspect so distances are not visually distorted
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.update_layout(width=900, height=600)
fig.show()

### 2b. Overlay the MRT network (stations + edges)

I had Claude overlay the location of MRT stations on top of the houses. The metro location and linkage data was already provided in the auxiliary dataset.

The houses are blue, the railway is red.

- Nodes: `data/features/raw/raw_graph_data/station-keys.csv` (`NAME`, `latitude`, `longitude`).
- Edges: `data/features/raw/raw_graph_data/sg-mrt-edges-weighted.csv` (`src_station`, `dst_station`,
  `distance_km`), where the station names reference `NAME`.

In [6]:
stations = pd.read_csv(STATION_KEYS_PATH)
mrt_edges = pd.read_csv(MRT_EDGES_PATH)
test_house = pd.read_csv(TEST_HOUSE_PATH)
print(f"Loaded {len(stations)} stations, {len(mrt_edges)} edges, {len(test_house):,} test houses")


def plot_houses_with_mrt(
    house_df,
    stations,
    mrt_edges,
    test_df=None,
    title="House locations with MRT network",
    lat_range=SG_LAT_RANGE,
    lon_range=SG_LON_RANGE,
):
    """Scatter of houses with the MRT stations (red markers) and edges (red lines) overlaid.

    house_df   : DataFrame with 'LATITUDE'/'LONGITUDE' columns (train houses, base layer).
    stations   : DataFrame with 'NAME', 'latitude', 'longitude' (station nodes).
    mrt_edges  : DataFrame with 'src_station'/'dst_station' referencing station NAME.
    test_df    : optional DataFrame with 'LATITUDE'/'LONGITUDE'; drawn in green.
    """
    # Base layer: train house points
    fig = px.scatter(
        house_df,
        x="LONGITUDE",
        y="LATITUDE",
        opacity=0.35,
        title=title,
        labels={"LONGITUDE": "Longitude", "LATITUDE": "Latitude"},
    )
    fig.update_traces(marker=dict(size=4, color="steelblue"), name="train houses", showlegend=True)

    # Test houses in green
    if test_df is not None:
        fig.add_scatter(
            x=test_df["LONGITUDE"],
            y=test_df["LATITUDE"],
            mode="markers",
            marker=dict(size=4, color="green"),
            opacity=0.35,
            name="test houses",
        )

    # Station coordinate lookup by name (station NAMEs are unique)
    coord = stations.set_index("NAME")[["latitude", "longitude"]]

    # Build all edges as ONE line trace: segments separated by None so plotly lifts the pen
    edge_lon, edge_lat = [], []
    missing = 0
    for src, dst in zip(mrt_edges["src_station"], mrt_edges["dst_station"]):
        if src in coord.index and dst in coord.index:
            edge_lon += [coord.at[src, "longitude"], coord.at[dst, "longitude"], None]
            edge_lat += [coord.at[src, "latitude"], coord.at[dst, "latitude"], None]
        else:
            missing += 1
    if missing:
        print(f"{missing} edge(s) skipped — station name not found in station-keys")

    # Edges under stations so the station markers sit on top of the lines
    fig.add_scatter(
        x=edge_lon,
        y=edge_lat,
        mode="lines",
        line=dict(color="red", width=1),
        opacity=0.7,
        name="MRT lines",
    )
    fig.add_scatter(
        x=stations["longitude"],
        y=stations["latitude"],
        mode="markers",
        marker=dict(color="red", size=6),
        name="MRT stations",
        text=stations["NAME"],
        hoverinfo="text",
    )

    fig.update_xaxes(range=list(lon_range))
    fig.update_yaxes(range=list(lat_range))
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.update_layout(width=950, height=650, legend_title_text="")
    return fig


fig = plot_houses_with_mrt(train_house, stations, mrt_edges, test_df=test_house)
fig.show()

Loaded 192 stations, 224 edges, 50,000 test houses


## 3. DBSCAN clustering

We run DBScan here. Tunable parameters and justification for my choice:
- **`EPS_KM`** — maximum neighbour distance (Haversine, in km). Two points are neighbours if within this distance. The logic here is that generally houses need to be within walking distance to share similar characteristics, which I determine to not exceed 500m.
- **`MIN_SAMPLES`** — minimum number of points (including the point itself) within `EPS_KM` for a point to be a core point. The Changi Village cluster only has 3 rows so we need to set this to be 3 or less.

Points not assigned to any cluster are labelled `-1` (noise). 

For tuning the parameters, I just had Claude plot the output. So long as there are no clear overlaps between the clusters I accept the parameter output.
Since `MIN_SAMPLES` already has a very low cutoff, I just leave it at 3 and only tune the `EPS_KM`


## Tune Results

|EPS_KM|Evaluation|
|---|---|
| 0.1 | Too much noise |
| 0.2 | The clusters have become very fine. Some clusters have enclaves within them |
| 0.3 | Clean segmentation, does not have as many small clusters compared to EPS = 0.2 and some areas are segmented correctly compared to EPS = 0.4 and EPS = 0.5 |
| 0.4 | Segmentation is cleaner, but some regions still not segmented properly (See Northwest, Chua Chu Kang area, there are 3 clusters but only 2 were identified) |
| 0.5 | There are some mini-clusters that should have been formed within some clusters (See Sembawang/North, there are 3 clusters but the scan only produces 2)|

I will be using the EPS_KM = 0.3 implementation for the modelling.

In [7]:
# Helper functions
def plot_house_clusters(
    coords_latlon,
    cluster_ids,
    title="DBSCAN clusters",
    lat_range=SG_LAT_RANGE,
    lon_range=SG_LON_RANGE,
    show_noise=True,
    top_n=None,
):
    """Scatter-plot houses coloured by cluster id.

    coords_latlon : array-like of shape (N, 2) as (latitude, longitude) in degrees,
                    or a DataFrame containing 'LATITUDE' and 'LONGITUDE' columns.
    dbscan_cluster   : array-like of length N with the cluster label per house
                    (-1 denotes noise).
    top_n         : if set, only the top_n largest clusters are coloured; smaller
                    clusters and noise are drawn as grey background, and the count
                    of houses covered by the top_n clusters is printed.
    show_noise    : whether to draw noise / non-top points as grey background.
    """
    if isinstance(coords_latlon, pd.DataFrame):
        lat = coords_latlon["LATITUDE"].to_numpy()
        lon = coords_latlon["LONGITUDE"].to_numpy()
    else:
        arr = np.asarray(coords_latlon, dtype=float)
        lat, lon = arr[:, 0], arr[:, 1]

    df = pd.DataFrame({
        "LATITUDE": lat,
        "LONGITUDE": lon,
        "dbscan_cluster": np.asarray(cluster_ids),
    })
    n_total = len(df)

    # Sizes of real clusters (exclude noise), largest first
    cluster_sizes = df.loc[df["dbscan_cluster"] != -1, "dbscan_cluster"].value_counts()

    # Decide which clusters get their own colour
    if top_n is not None:
        kept = cluster_sizes.head(top_n).index.tolist()
        covered = int(cluster_sizes.loc[kept].sum())
        n_clustered = int(cluster_sizes.sum())
        pct_all = covered / n_total if n_total else 0.0
        pct_clustered = covered / n_clustered if n_clustered else 0.0
        print(
            f"Top {top_n} clusters cover {covered:,} of {n_total:,} houses "
            f"({pct_all:.1%} of all houses; {pct_clustered:.1%} of clustered houses)."
        )
    else:
        kept = cluster_sizes.index.tolist()

    in_top = df["dbscan_cluster"].isin(kept)
    clustered = df[in_top].copy()
    clustered["cluster_label"] = clustered["dbscan_cluster"].astype(str)
    background = df[~in_top]  # smaller clusters + noise

    order = [str(c) for c in sorted(kept)]
    fig = px.scatter(
        clustered,
        x="LONGITUDE",
        y="LATITUDE",
        color="cluster_label",
        category_orders={"cluster_label": order},
        opacity=0.7,
        title=title,
        labels={"LONGITUDE": "Longitude", "LATITUDE": "Latitude", "cluster_label": "cluster"},
    )
    fig.update_traces(marker=dict(size=4))

    if show_noise and len(background):
        label = "other clusters / noise" if top_n is not None else "noise (-1)"
        fig.add_scatter(
            x=background["LONGITUDE"],
            y=background["LATITUDE"],
            mode="markers",
            marker=dict(size=3, color="lightgrey"),
            name=label,
            opacity=0.4,
        )

    fig.update_xaxes(range=list(lon_range))
    fig.update_yaxes(range=list(lat_range))
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.update_layout(width=950, height=650, legend_title_text="cluster")
    return fig

def save_dbscan(dbs, path):
    core_points = dbs.core_sample_indices_
    core_coords = dbs.components_
    eps = np.array([dbs.eps])
    np.savez(f'{path}/dbscan_output.npz', core_points, core_coords, eps)
    joblib.dump(dbs, f'{path}/dbscan_object.pkl')

In [8]:
round(0.3001,1)

0.3

In [10]:
params = list(np.linspace(0.1,0.5,num=5))
print(params)
MIN_SAMPLES = 3
for eps in params:
    EPS_KM = round(eps, 1)
    print('=================================================')
    # Try loading DBScan object
    try:
        print(f'Loading DBScan object found for EPS = {EPS_KM}km...')
        coords_rad = np.radians(train_house[["LATITUDE", "LONGITUDE"]].to_numpy())
        db = joblib.load(f'{DBSCAN_PATH}/eps_{int(EPS_KM*1000)}m.pkl')

    except:
        print(f'No DBScan object found for EPS = {EPS_KM}km, training')
        eps_radians = EPS_KM / EARTH_RADIUS_KM
        coords_rad = np.radians(train_house[["LATITUDE", "LONGITUDE"]].to_numpy())
        db = DBSCAN(
            eps=eps_radians,
            min_samples=MIN_SAMPLES,
            metric="haversine",
            algorithm="ball_tree",
        )
    train_house["dbscan_cluster"] = db.fit_predict(coords_rad)

    # Summary of clustering result
    labels = train_house["dbscan_cluster"]
    n_clusters = labels[labels != -1].nunique()
    n_noise = int((labels == -1).sum())

    print(f"EPS_KM = {EPS_KM}, MIN_SAMPLES = {MIN_SAMPLES}")
    print(f"Clusters found : {n_clusters}")
    print(f"Noise points   : {n_noise:,} ({n_noise / len(train_house):.1%})")
    # print("\nTop cluster sizes:")
    # display(labels[labels != -1].value_counts().head(10))

    # Full clustering (all clusters coloured)
    fig = plot_house_clusters(
        train_house[["LATITUDE", "LONGITUDE"]],
        train_house["dbscan_cluster"],
        title=f"DBSCAN clusters (EPS_KM={EPS_KM}, MIN_SAMPLES={MIN_SAMPLES})",
    )
    fig.show()
    # joblib.dump(db, f'{DBSCAN_PATH}/eps_{int(EPS_KM*1000)}m.pkl')

[0.1, 0.2, 0.30000000000000004, 0.4, 0.5]
Loading DBScan object found for EPS = 0.1km...
EPS_KM = 0.1, MIN_SAMPLES = 3
Clusters found : 461
Noise points   : 364 (3.9%)


Loading DBScan object found for EPS = 0.2km...
EPS_KM = 0.2, MIN_SAMPLES = 3
Clusters found : 85
Noise points   : 17 (0.2%)


Loading DBScan object found for EPS = 0.3km...
EPS_KM = 0.3, MIN_SAMPLES = 3
Clusters found : 47
Noise points   : 5 (0.1%)


Loading DBScan object found for EPS = 0.4km...
EPS_KM = 0.4, MIN_SAMPLES = 3
Clusters found : 30
Noise points   : 4 (0.0%)


Loading DBScan object found for EPS = 0.5km...
EPS_KM = 0.5, MIN_SAMPLES = 3
Clusters found : 20
Noise points   : 3 (0.0%)


## 4. Predict clusters for the test dataset

The following is an output from Claude: 

>sklearn's `DBSCAN` has no `predict` method — it is transductive. To label the
>**test** houses we apply DBSCAN's own reachability rule to new points:
> a test point joins the cluster of its **nearest core point**, provided that core
> point is within `eps` (the min-distance); otherwise it is labelled `-1` (noise).

>This reuses only the fitted object's **core points** (`db.components_`, with their
>labels) and its **`eps`** parameter, so the test assignment is consistent with how
>the training clusters were formed. Core points are stored in the same radian space
>the model was fitted in, so test coordinates are converted to radians and compared
>with the Haversine metric.

In short, we assign the class of a test data to the class of its nearest core point.

In [84]:
def dbscan_predict(db, test_coords_latlon):
    """Assign cluster ids to new points using a fitted DBSCAN's core points and eps.

    db                 : a fitted sklearn DBSCAN (metric='haversine'); its
                         `components_` (core points, in radians) and `eps`
                         (min-distance, in radians) drive the assignment.
    test_coords_latlon : array-like (N, 2) as (latitude, longitude) in degrees, or a
                         DataFrame with 'LATITUDE'/'LONGITUDE' columns.

    Returns an int array of length N: the nearest core point's cluster label when
    that core point is within `db.eps`, else -1 (noise).
    """
    if isinstance(test_coords_latlon, pd.DataFrame):
        test_arr = test_coords_latlon[["LATITUDE", "LONGITUDE"]].to_numpy()
    else:
        test_arr = np.asarray(test_coords_latlon, dtype=float)

    core_coords = db.components_                       # core points, already in radians
    core_labels = db.labels_[db.core_sample_indices_]  # cluster id of each core point

    if len(core_coords) == 0:  # no core points -> everything is noise
        return np.full(len(test_arr), -1, dtype=int)

    test_rad = np.radians(test_arr)
    nn = NearestNeighbors(n_neighbors=1, metric="haversine", algorithm="ball_tree")
    nn.fit(core_coords)
    dist, idx = nn.kneighbors(test_rad)
    dist, idx = dist[:, 0], idx[:, 0]

    pred = core_labels[idx].astype(int)
    pred[dist > db.eps] = -1  # nearest core point is beyond the min-distance -> noise
    return pred

In [85]:
EPS_KM = 0.3
train_house = pd.read_csv(TRAIN_HOUSE_PATH)[['TOWN','BLOCK','STREET','POSTAL_CODE','LATITUDE','LONGITUDE']].drop_duplicates()
test_house = pd.read_csv(TEST_HOUSE_PATH)[['TOWN','BLOCK','STREET','POSTAL_CODE','LATITUDE','LONGITUDE']].drop_duplicates()
db = joblib.load(f'{DBSCAN_PATH}/eps_{int(EPS_KM*1000)}m.pkl')

train_house["dbscan_cluster"] = dbscan_predict(db, train_house)
test_house["dbscan_cluster"] = dbscan_predict(db, test_house)
test_labels = test_house["dbscan_cluster"]
test_noise = int((test_labels == -1).sum())
print(f"Test houses           : {len(test_house):,}")
print(f"Assigned to a cluster : {(test_labels != -1).sum():,} "
      f"({(test_labels != -1).mean():.1%})")
print(f"Noise (-1)            : {test_noise:,} ({test_noise / len(test_house):.1%})")
print(f"Distinct clusters hit : {test_labels[test_labels != -1].nunique()}")

# Visual check that the test assignment respects the training cluster geography
print("=========== TRAIN ===========")
fig = plot_house_clusters(
    train_house[["LATITUDE", "LONGITUDE"]],
    train_house["dbscan_cluster"],
    title=f"Predicted train clusters (EPS_KM={EPS_KM}, MIN_SAMPLES={MIN_SAMPLES})",
)
fig.show()
print("=========== TEST ===========")
fig = plot_house_clusters(
    test_house[["LATITUDE", "LONGITUDE"]],
    test_house["dbscan_cluster"],
    title=f"Predicted test clusters (EPS_KM={EPS_KM}, MIN_SAMPLES={MIN_SAMPLES})",
)
fig.show()

Test houses           : 8,940
Assigned to a cluster : 8,935 (99.9%)
Noise (-1)            : 5 (0.1%)
Distinct clusters hit : 47
=========== TRAIN ===========


=========== TEST ===========


In [86]:
train_house.to_csv(f'{CLUSTER_OUTPUT_PATH}/train_dbscan_cluster.csv', index = False)
test_house.to_csv(f'{CLUSTER_OUTPUT_PATH}/test_dbscan_cluster.csv', index = False)